In [1]:
!pip install trl transformers accelerate peft datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00


In [2]:
import json, re
import torch
import torch.nn as nn
import torch.ao.quantization as tq
import wandb
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainerCallback, TrainerState, TrainerControl, BitsAndBytesConfig
)
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from peft import PeftModel

# ── Config ───────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

TRAIN_PATH    = "/kaggle/input/datasets/mythreyee1006/train-dataset-abdomen-new/train_records_abdomenCT.jsonl"
TEST_PATH     = "/kaggle/input/datasets/mythreyee1006/test-dataset-abdomen-final/test_records_abdomenCT.jsonl"
FEW_SHOT_PATH = "/kaggle/input/datasets/mythreyee1006/fewshot-abdomen/few_shot_abd_ct.json"

EVAL_BATCH_SIZE = 8
EVAL_STEPS = 50

SYSTEM_PROMPT = (
    "You are a clinical assistant. Extract the exact sentence(s) containing "
    "incidental findings from the report. If none are present, return an empty list. "
    "Always respond with valid JSON in the format: "
    '{"contains_IF": true/false, "incidental_sentences": []}.'
)

DATASET_TAG = "abdomen"

# ── wandb ────────────────────────────────────────────────────────────────
wandb_key="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" # replace with your key or set WANDB_API_KEY env var

# ── Data loading (already merged jsonl, messages format) ──────────────────
def load_jsonl_records(path):
    """Reads the pre-merged jsonl (messages format) and converts to
    {report_id, free_text, gold} dicts to match the eval functions."""
    records = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            rec  = json.loads(line)
            msgs = rec["messages"]
            free_text = msgs[1]["content"].replace("Report:\n", "").strip()
            gold = json.loads(msgs[2]["content"])
            records.append({
                "report_id": f"abd_{i}",
                "free_text": free_text,
                "gold": gold,
            })
    return records

def to_chatml_record(record):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Report:\n{record['free_text']}"},
            {"role": "assistant", "content": json.dumps({
                "contains_IF": record["gold"]["contains_IF"],
                "incidental_sentences": record["gold"]["incidental_sentences"],
            })},
        ]
    }

from sklearn.model_selection import train_test_split

# Use the existing loader — it already extracts free_text + gold correctly
train_raw = load_jsonl_records(TRAIN_PATH)
print(f"Loaded {len(train_raw)} structured records")
print(train_raw[0])  # sanity check shape: {report_id, free_text, gold}

labels = [r["gold"]["contains_IF"] for r in train_raw]

train_split, val_structured = train_test_split(
    train_raw,
    test_size=100,
    stratify=labels,
    random_state=42,
)
print(f"Train split: {len(train_split)}, Val (structured): {len(val_structured)}")

from collections import Counter
print("Train label dist:", Counter([r["gold"]["contains_IF"] for r in train_split]))
print("Val label dist:", Counter([r["gold"]["contains_IF"] for r in val_structured]))

train_chatml = [to_chatml_record(r) for r in train_split]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def format_chat(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return example

train_dataset = Dataset.from_list(train_chatml).map(format_chat)
print(f"Train dataset ready: {len(train_dataset)}")

Loaded 1184 structured records
{'report_id': 'abd_0', 'free_text': "No priors available for comparison. Today's examination is markedly", 'gold': {'contains_IF': False, 'incidental_sentences': []}}
Train split: 1084, Val (structured): 100
Train label dist: Counter({False: 664, True: 420})
Val label dist: Counter({False: 61, True: 39})


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1084 [00:00<?, ? examples/s]

Train dataset ready: 1084


In [3]:
import re


def parse_output(raw_text):
    def _try(text):
        try:
            val = json.loads(text)
            return val if isinstance(val, dict) else None
        except json.JSONDecodeError:
            return None

    result = _try(raw_text)
    if result is not None:
        return result

    match = re.search(r'\{.*\}', raw_text, re.DOTALL)
    if match:
        result = _try(match.group())
        if result is not None:
            return result

    return None

def build_messages(record, few_shot_pool=None, n_shot=0):
    """Build the chat message list for a single eval record.

    record: dict with at least "free_text" (and "gold" for few-shot examples).
    few_shot_pool: list of records with the same shape, used to draw few-shot
        examples from (should NOT overlap with the eval set).
    n_shot: number of few-shot examples to prepend.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if few_shot_pool and n_shot > 0:
        for ex in few_shot_pool[:n_shot]:
            messages.append({"role": "user", "content": f"Report:\n{ex['free_text']}"})
            messages.append({
                "role": "assistant",
                "content": json.dumps({
                    "contains_IF": ex["gold"]["contains_IF"],
                    "incidental_sentences": ex["gold"]["incidental_sentences"],
                }),
            })

    messages.append({"role": "user", "content": f"Report:\n{record['free_text']}"})
    return messages


def run_eval(model, tokenizer, records, few_shot_pool=None, n_shot=0, n=None,
             desc="", batch_size=EVAL_BATCH_SIZE):
    import transformers
    transformers.logging.set_verbosity_error()
    model.eval()

    subset = records[:n] if n else records
    total_tp = total_fp = total_fn = 0
    neg_scores, pos_scores = [], []

    for start in range(0, len(subset), batch_size):
        batch_records = subset[start:start + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                build_messages(r, few_shot_pool, n_shot),
                tokenize=False, add_generation_prompt=True
            )
            for r in batch_records
        ]

        inputs = tokenizer(
            texts, return_tensors="pt", padding=True, truncation=True
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        for i, record in enumerate(batch_records):
            generated = output_ids[i][prompt_len:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            parsed = parse_output(raw)

            gold_set = set(record["gold"]["incidental_sentences"])
            pred_set = set(parsed.get("incidental_sentences", [])) if parsed else set()

            if len(gold_set) == 0:
                neg_scores.append(1.0 if len(pred_set) == 0 else 0.0)
            else:
                tp = len(gold_set & pred_set)
                fp = len(pred_set - gold_set)
                fn = len(gold_set - pred_set)
                total_tp += tp
                total_fp += fp
                total_fn += fn
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
                pos_scores.append(f1)

    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    macro_f1 = (avg_neg + avg_pos) / 2 if (neg_scores and pos_scores) else (avg_neg or avg_pos)

    n_neg, n_pos = len(neg_scores), len(pos_scores)
    weighted_f1 = (n_neg * avg_neg + n_pos * avg_pos) / (n_neg + n_pos) if (n_neg + n_pos) > 0 else 0.0

    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0.0

    model.train()
    return {
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "weighted_f1": round(weighted_f1, 4),
    }


class MacroF1EarlyStoppingCallback(TrainerCallback):
    def __init__(self, eval_records, tokenizer, run_name, eval_steps=EVAL_STEPS, patience=3):
        self.eval_records = eval_records
        self.tokenizer = tokenizer
        self.run_name = run_name
        self.eval_steps = eval_steps
        self.patience = patience
        self.best_f1 = -1.0
        self.no_improve = 0
        self.best_step = 0
        self.best_metrics = None
        self.best_ckpt_dir = f"./qlora_best_{run_name}"

    def on_step_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        if state.global_step % self.eval_steps != 0 or state.global_step == 0:
            return control

        model = kwargs["model"]
        metrics = run_eval(model, self.tokenizer, self.eval_records, n=100, desc="val")

        print(f"\nStep {state.global_step} | Macro F1: {metrics['macro_f1']:.4f} | "
              f"Micro F1: {metrics['micro_f1']:.4f} | Weighted F1: {metrics['weighted_f1']:.4f} | "
              f"Best: {self.best_f1:.4f}")
        wandb.log({
            "val_macro_f1": metrics["macro_f1"],
            "val_micro_f1": metrics["micro_f1"],
            "val_weighted_f1": metrics["weighted_f1"],
            "step": state.global_step,
        })

        if metrics["macro_f1"] > self.best_f1:
            self.best_f1 = metrics["macro_f1"]
            self.best_step = state.global_step
            self.best_metrics = metrics
            self.no_improve = 0
            model.save_pretrained(self.best_ckpt_dir)
            self.tokenizer.save_pretrained(self.best_ckpt_dir)
            print(f"  New best saved -> {self.best_ckpt_dir}")
        else:
            self.no_improve += 1
            print(f"  No improvement ({self.no_improve}/{self.patience})")

        if self.no_improve >= self.patience:
            print(f"\nEarly stopping at step {state.global_step}. "
                  f"Best step {self.best_step}, macro F1={self.best_f1:.4f}")
            control.should_training_stop = True

        return control


In [4]:
if wandb_key:
    wandb.login(key=wandb_key)
else:
    os.environ["WANDB_MODE"] = "disabled"  # or "offline" to log locally without a key
    print("WANDB_API_KEY not set — running with wandb disabled.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mythreyee2006 (mythreyee2006-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# Hyperparameter grid
lora_configs = [
    {"r": 8, "lora_alpha": 16},
]

sweep_results = []

# int8 quantization config, shared across sweep runs. This is what actually
# makes the loaded model comparable to a GGUF Q8_0 QLoRA baseline — the
# previous `torch_dtype=torch.int8` line did NOT do real quantization, it
# just cast raw weight values into int8 range with no scale/zero-point.
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

for config in lora_configs:
    run_name = f"r{config['r']}_alpha{config['lora_alpha']}"
    print(f"\n{'=' * 50}")
    print(f"Starting run: {run_name}")
    print(f"{'=' * 50}")

    wandb.init(
        project="incidental-findings-finetuning",
        name=run_name,
        config=config,
        reinit=True,
    )

    # Reload a fresh base model, quantized to int8, for each run.
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map={"": 0},
    )
    model = prepare_model_for_kbit_training(model)

    peft_config = LoraConfig(
        r=config["r"],
        lora_alpha=config["lora_alpha"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    training_args = SFTConfig(
        output_dir=f"./qlora_best_{run_name}",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        num_train_epochs=100,        # high ceiling, early stopping will trigger
        save_strategy="steps",
        save_steps=50,
        eval_strategy="no",          # we handle eval in the callback
        lr_scheduler_type="cosine",
        warmup_steps=10,
        fp16=False,                  # base weights are int8; LoRA adapters stay fp32/bf16
        bf16=False,
        report_to="wandb",
        dataset_text_field="text",
        gradient_checkpointing=True,
    )

    callback = MacroF1EarlyStoppingCallback(
        eval_records=val_structured,
        tokenizer=tokenizer,
        run_name=run_name,
        eval_steps=50,
        patience=3,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_args,
        callbacks=[callback],
    )

    trainer.train()

    sweep_results.append({
        "run": run_name,
        "r": config["r"],
        "lora_alpha": config["lora_alpha"],
        "best_f1": callback.best_f1,
        "best_step": callback.best_step,
    })

    wandb.finish()

# Summary
print("\n" + "=" * 50)
print("Sweep Summary:")
print("=" * 50)
for r in sorted(sweep_results, key=lambda x: x["best_f1"], reverse=True):
    print(f"  {r['run']:20s} | Best F1: {r['best_f1']:.4f} | Step: {r['best_step']}")



Starting run: r8_alpha16


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260707_093047-mbt8uhmo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run r8_alpha16
wandb: ⭐️ View project at https://wandb.ai/mythreyee2006-indian-institute-of-technology-madras/incidental-findings-finetuning
wandb: 🚀 View run at https://wandb.ai/mythreyee2006-indian-institute-of-technology-madras/incidental-findings-finetuning/runs/mbt8uhmo


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...


Step 650 | Macro F1: 0.5410 | Micro F1: 0.1918 | Weighted F1: 0.6420 | Best: 0.5577
  No improvement (2/3)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



Step 700 | Macro F1: 0.5299 | Micro F1: 0.1449 | Weighted F1: 0.6333 | Best: 0.5577
  No improvement (3/3)

Early stopping at step 700. Best step 550, macro F1=0.5577


wandb: updating run metadata
wandb: uploading history steps 82-84, summary, console lines 68-72
wandb: 
wandb: Run history:
wandb:                      step ▁▂▂▃▃▄▄▅▅▆▆▇▇█
wandb:             train/entropy █▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
wandb:               train/epoch ▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█
wandb:         train/global_step ▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█
wandb:           train/grad_norm █▇▂▁▁▂▂▂▂▂▂▂▂▃▃▃▃▂▃▃▃▃▃▃▃▄▃▃▄▃▄▄▄▄▄▄▄▄▄▄
wandb:       train/learning_rate █████████▇▇▇▇▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▃▃▃▃▂▂▂▁▁
wandb:                train/loss █▅▄▄▄▃▃▃▃▃▃▃▂▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁
wandb: train/mean_token_accuracy ▁▃▅▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████████
wandb:          train/num_tokens ▁▁▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█
wandb:              val_macro_f1 ▁▁▂▁▂▃▃▆▅▃█▄▆▅
wandb:                        +2 ...
wandb: 
wandb: Run summary:
wandb:                      step 700
wandb:                total_flos 1.8412294660598784e+16
wandb:             train/entro


Sweep Summary:
  r8_alpha16           | Best F1: 0.5577 | Step: 550


In [6]:
# %% [code]
# Standalone evaluation script.
# Run this AFTER training — it does not train anything, it just loads a
# saved LoRA checkpoint and scores it against the held-out test set.

checkpoint_dir = "./qlora_best_r8_alpha16"

def load_test_structured_records(path):
    """Reads the combined abdomen test jsonl (messages format) and converts
    to {report_id, free_text, gold} dicts matching the eval pipeline."""
    records = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            rec  = json.loads(line)
            msgs = rec["messages"]
            free_text = msgs[1]["content"].replace("Report:\n", "").strip()
            gold = json.loads(msgs[2]["content"])
            records.append({
                "report_id": f"abd_test_{i}",
                "free_text": free_text,
                "gold": {
                    "contains_IF": gold["contains_IF"],
                    "incidental_sentences": gold["incidental_sentences"],
                },
            })
    return records

test_structured_records = load_test_structured_records(TEST_PATH)
print(f"Loaded {len(test_structured_records)} abdomen test records.")

# %% [code]
# --- Load tokenizer + quantized base model + LoRA adapter ---
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for correct batched generation

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
)
model = PeftModel.from_pretrained(base_model, checkpoint_dir)
model.eval()

# %% [code]
# --- Eval helpers ---

def parse_output(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{.*\}', raw_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None


def build_messages(record, few_shot_pool=None, n_shot=0):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if few_shot_pool and n_shot > 0:
        for ex in few_shot_pool[:n_shot]:
            messages.append({"role": "user", "content": f"Report:\n{ex['free_text']}"})
            messages.append({
                "role": "assistant",
                "content": json.dumps({
                    "contains_IF": ex["gold"]["contains_IF"],
                    "incidental_sentences": ex["gold"]["incidental_sentences"],
                }),
            })

    messages.append({"role": "user", "content": f"Report:\n{record['free_text']}"})
    return messages


def run_eval(model, tokenizer, records, few_shot_pool=None, n_shot=0, n=None,
             batch_size=EVAL_BATCH_SIZE):
    import transformers
    transformers.logging.set_verbosity_error()
    model.eval()

    subset = records[:n] if n else records
    total_tp = total_fp = total_fn = 0
    neg_scores, pos_scores = [], []

    for start in range(0, len(subset), batch_size):
        batch_records = subset[start:start + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                build_messages(r, few_shot_pool, n_shot),
                tokenize=False, add_generation_prompt=True
            )
            for r in batch_records
        ]

        inputs = tokenizer(
            texts, return_tensors="pt", padding=True, truncation=True
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        for i, record in enumerate(batch_records):
            generated = output_ids[i][prompt_len:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            parsed = parse_output(raw)

            gold_set = set(record["gold"]["incidental_sentences"])
            pred_set = set(parsed.get("incidental_sentences", [])) if parsed else set()

            if len(gold_set) == 0:
                neg_scores.append(1.0 if len(pred_set) == 0 else 0.0)
            else:
                tp = len(gold_set & pred_set)
                fp = len(pred_set - gold_set)
                fn = len(gold_set - pred_set)
                total_tp += tp
                total_fp += fp
                total_fn += fn
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
                pos_scores.append(f1)

    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    macro_f1 = (avg_neg + avg_pos) / 2 if (neg_scores and pos_scores) else (avg_neg or avg_pos)

    n_neg, n_pos = len(neg_scores), len(pos_scores)
    weighted_f1 = (n_neg * avg_neg + n_pos * avg_pos) / (n_neg + n_pos) if (n_neg + n_pos) > 0 else 0.0

    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0.0

    return {
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "weighted_f1": round(weighted_f1, 4),
        "n_negative": n_neg,
        "n_positive": n_pos,
    }

# %% [code]
# --- Run it ---
metrics = run_eval(model, tokenizer, test_structured_records, n=None)

print(f"\nTest set results ({len(test_structured_records)} records — "
      f"{metrics['n_negative']} no-finding, {metrics['n_positive']} with-finding):")
print(f"  Macro F1:    {metrics['macro_f1']:.4f}")
print(f"  Micro F1:    {metrics['micro_f1']:.4f}")
print(f"  Weighted F1: {metrics['weighted_f1']:.4f}")

Loaded 100 abdomen test records.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



Test set results (100 records — 61 no-finding, 39 with-finding):
  Macro F1:    0.5733
  Micro F1:    0.2740
  Weighted F1: 0.6563
